# Website Crawler & Text Extractor

This notebook lets you **crawl websites** and **save their content as text files**. Just follow the steps below — fill in your settings, then run each cell in order.

## Step 1: Set Up (Run Once)

Run the cell below **once** to install the required tools. You can skip this step next time.

In [ ]:
# Run this cell once to set up. You can skip it next time.
# %pip install -e ..
# !crawl4ai-setup
# !playwright install --with-deps chromium

## Step 2: Enter Your Settings

Edit the values below to match what you want to crawl, then **run the cell**.

In [ ]:
# --- Your URLs (comma-separated) ---
urls = "https://www.starhub.com/personal.html"

# --- Crawler options ---
exclude_paths = ""        # URLs matching these patterns will be skipped (leave empty for none)
include_only_paths = "/personal"   # Only crawl URLs matching these patterns (leave empty for all)
limit = 999999            # Maximum number of pages to crawl
max_depth = 5             # How many clicks deep to follow links
flush_interval = 1       # Save urls.txt to disk every N pages (protects against crashes)
delay = 5                 # Seconds to wait between each page (helps avoid bot detection)
stealth = True            # Avoid bot detection (simulates real browser behavior)
max_retries = 3           # Retry rounds for blocked pages (0 = no retries)

# --- Page options ---
exclude_tags = "nav, script, form, style"   # HTML elements to remove
include_only_tags = ""                       # Only keep content inside these elements (leave empty for all)
wait_for = 12                              # Seconds to wait after page loads (set a number for slow sites)
timeout = 30000                              # Max wait time per page (milliseconds)
max_file_size_mb = 15                        # Max size of each output file (MB)
extract_main_content = True                  # True = main content only, False = full page
output_extension = ".md"                    # File format: ".txt" or ".md"
separate_items = False                       # Add --- lines between repeating items (e.g. product listings)
item_selector = ""                           # CSS selector to target specific items (leave empty for auto-detection)

print("Settings saved!")

## Step 3: Start Crawling

Run the cell below to start crawling. Content is **extracted and saved to disk automatically** as pages are crawled — if the process is interrupted, your progress is safe.

In [ ]:
from crawl4md import SiteCrawler, CrawlerConfig, PageConfig, ContentExtractor, FileWriter

crawler_config = CrawlerConfig(
    urls=urls,
    exclude_paths=exclude_paths,
    include_only_paths=include_only_paths,
    limit=limit,
    max_depth=max_depth,
    flush_interval=flush_interval,
    delay=delay,
    stealth=stealth,
    max_retries=max_retries,
)
page_config = PageConfig(
    exclude_tags=exclude_tags,
    include_only_tags=include_only_tags,
    wait_for=wait_for,
    timeout=timeout,
    max_file_size_mb=max_file_size_mb,
    extract_main_content=extract_main_content,
    output_extension=output_extension,
    separate_items=separate_items,
    item_selector=item_selector,
)

extractor = ContentExtractor(page_config)
writer = FileWriter(max_file_size_mb=page_config.max_file_size_mb, file_extension=page_config.output_extension)

crawler = SiteCrawler(crawler_config, page_config, extractor=extractor, writer=writer)
results = crawler.crawl()

## Step 4: View Results

Run the cell below to see a summary of the saved files.

In [ ]:
from pathlib import Path
import re

output_dir = crawler.output_dir
success_count = sum(1 for r in results if r.success)
fail_count = sum(1 for r in results if not r.success)

print(f"Results: {success_count} succeeded, {fail_count} failed")
print(f"Output folder: {output_dir}\n")

def _print_files(label, paths):
    if not paths:
        return
    print(f"  {label}:")
    for f in paths:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"    {f.name} ({size_mb:.2f} MB)")
    print()

# --- Per-round files ---
round_nums = sorted({
    int(m.group(1))
    for f in output_dir.iterdir()
    if (m := re.match(r"round_(\d+)_", f.name))
})
for rn in round_nums:
    prefix = f"round_{rn}_"
    content = sorted(output_dir.glob(f"{prefix}success_content_*"))
    fail_content = sorted(output_dir.glob(f"{prefix}fail_content_*"))
    url_files = sorted(output_dir.glob(f"{prefix}*urls*.txt"))
    print(f"--- Round {rn} ---")
    _print_files("Success content", content)
    _print_files("Fail content", fail_content)
    _print_files("URL lists", url_files)

# --- Final merged files (unsorted) ---
final_success = sorted(output_dir.glob("final_success_content_*"))
final_fail = sorted(output_dir.glob("final_fail_content_*"))
final_urls = sorted(output_dir.glob("final_*_urls.txt"))
if final_success or final_fail:
    print("--- Final (unsorted) ---")
    _print_files("Success content", final_success)
    _print_files("Fail content", final_fail)
    _print_files("URL lists", final_urls)

# --- Sorted files (primary output) ---
sorted_success = sorted(output_dir.glob("sorted_final_success_content_*"))
sorted_fail = sorted(output_dir.glob("sorted_final_fail_content_*"))
sorted_urls = sorted(output_dir.glob("sorted_final_*_urls.txt"))
if sorted_success or sorted_fail:
    print("--- Sorted by URL path (primary output) ---")
    _print_files("Success content", sorted_success)
    _print_files("Fail content", sorted_fail)
    _print_files("URL lists", sorted_urls)

if fail_count > 0:
    print(f"See sorted_final_fail_urls.txt for the {fail_count} URL(s) that could not be crawled.")